# ResNet‑18 Feature Maps and Image Embedding (Full Explanation)

This notebook explains **how ResNet‑18 processes an image step by step**, showing:
- what feature maps represent at each depth
- how abstraction evolves from edges to semantics
- how the final 512‑D embedding is formed and exported


## Imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
from torchvision import models, transforms
from PIL import Image
from pathlib import Path
from matplotlib.patches import FancyArrowPatch

In [ ]:
device = torch.device('cpu')
print('Using device:', device)

## Load ResNet‑18 in embedding mode

The final classification layer is removed so the network outputs a **512‑dimensional feature embedding** instead of class predictions.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Identity()
model.eval()

## Feature‑map hooks

Hooks capture the **internal representations** (feature maps) produced by convolutional layers during the forward pass.

In [ ]:
feature_maps = {}

def hook_fn(name):
    def hook(module, inp, out):
        feature_maps[name] = out.detach().cpu()
    return hook

model.conv1.register_forward_hook(hook_fn('conv1'))
model.layer1.register_forward_hook(hook_fn('layer1'))
model.layer2.register_forward_hook(hook_fn('layer2'))
model.layer3.register_forward_hook(hook_fn('layer3'))
model.layer4.register_forward_hook(hook_fn('layer4'))

## Image preprocessing

The image is resized, center‑cropped, converted to a tensor, and normalized using ImageNet statistics.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
img_path = Path('data') / '25DIV3FAY_1002.jpg'
assert img_path.exists(), 'Image not found in data folder'

img = Image.open(img_path).convert('RGB')
x = transform(img).unsqueeze(0)

plt.figure(figsize=(4,4))
plt.imshow(img)
plt.axis('off')
plt.title('Input Image')
plt.show()

In [ ]:
with torch.no_grad():
    embedding = model(x).squeeze().numpy()

print('Embedding shape:', embedding.shape)

## Feature‑map visualization utility

In [ ]:
def visualize_feature_maps(fmap, title, max_channels=16):
    fmap = fmap.squeeze(0)
    n = min(fmap.shape[0], max_channels)
    grid = math.ceil(math.sqrt(n))
    plt.figure(figsize=(8,8))
    for i in range(n):
        fm = fmap[i]
        fm = (fm - fm.min()) / (fm.max() - fm.min() + 1e-6)
        plt.subplot(grid, grid, i+1)
        plt.imshow(fm, cmap='viridis')
        plt.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

## conv1 — edges and color gradients

The first convolutional layer learns **basic visual primitives**:
- oriented edges
- color contrasts
- intensity changes

These resemble classical edge detectors.

In [ ]:
visualize_feature_maps(feature_maps['conv1'], 'conv1: edges and colors')

## layer1 — low‑level textures

This layer uses stacked **3×3, stride‑1 convolutions** to combine edges into textures:
- leaf patterns
- grass granularity
- fine repeated motifs

In [ ]:
visualize_feature_maps(feature_maps['layer1'], 'layer1: textures')

## layer2 — canopy patterns and mid‑level parts

Textures are grouped into **mid‑level structures**, such as canopy clusters and object parts.

In [ ]:
visualize_feature_maps(feature_maps['layer2'], 'layer2: canopy patterns')

## layer3 — spatial structure

This layer captures **spatial arrangement and layout**, responding to structure rather than texture.

In [ ]:
visualize_feature_maps(feature_maps['layer3'], 'layer3: spatial structure')

## layer4 — high‑level semantics

Feature maps at this depth encode **abstract, class‑relevant concepts**. These are no longer human‑interpretable, and that is expected.

In [ ]:
visualize_feature_maps(feature_maps['layer4'], 'layer4: high-level semantics')